# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [176]:
# Step 0 - Import libraries
# --------------------------

# Import the libraries needed for this section
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier

# For evaluating Regression models (Part 1)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# For evaluating Classification models (Part 2)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# For Decision Trees
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree

# For Random Forest
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

# Standard seed used throughout the notebook, for reproducibility
SEED = 1

In [177]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection

In [178]:
spaceship.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [179]:
spaceship.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [180]:
spaceship.describe()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,8514.000000,8512.000000,8510.000000,8485.000000,8510.000000,8505.000000
mean,28.827930,224.687617,458.077203,173.729169,311.138778,304.854791
std,14.489021,666.717663,1611.489240,604.696458,1136.705535,1145.717189
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,47.000000,76.000000,27.000000,59.000000,46.000000
max,79.000000,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [181]:
spaceship.shape

(8693, 14)

In [182]:
spaceship["Transported"].value_counts()

Transported
True     4378
False    4315
Name: count, dtype: int64

In [183]:
# Step 6 - Extract Cabin Deck

spaceship["Cabin"] = spaceship["Cabin"].str[0]

spaceship["Cabin"].value_counts(dropna=False)

Cabin
F      2794
G      2559
E       876
B       779
C       747
D       478
A       256
NaN     199
T         5
Name: count, dtype: int64

In [184]:
# Step 7 - Fill missing categorical values

categorical_columns = [
    "HomePlanet",
    "CryoSleep",
    "Destination",
    "VIP",
    "Cabin"
]

for col in categorical_columns:
    spaceship[col] = spaceship[col].fillna(
        spaceship[col].mode()[0]
    )

spaceship[categorical_columns].isnull().sum()

C:\Users\Winifred\AppData\Local\Temp\ipykernel_11076\600100862.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  spaceship[col] = spaceship[col].fillna(


HomePlanet     0
CryoSleep      0
Destination    0
VIP            0
Cabin          0
dtype: int64

In [185]:
# Step 8 - Fill missing numerical values

numerical_columns = [
    "Age",
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck"
]

for col in numerical_columns:
    spaceship[col] = spaceship[col].fillna(
        spaceship[col].median()
    )

spaceship[numerical_columns].isnull().sum()

Age             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
dtype: int64

In [186]:
# Step 9 - Final missing-value check

spaceship.isnull().sum()

PassengerId       0
HomePlanet        0
CryoSleep         0
Cabin             0
Destination       0
Age               0
VIP               0
RoomService       0
FoodCourt         0
ShoppingMall      0
Spa               0
VRDeck            0
Name            200
Transported       0
dtype: int64

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

In [187]:
# Step 10 - Create X and y

y = spaceship["Transported"].astype(int)

X = spaceship.drop(
    columns=["Transported", "PassengerId", "Name"]
)

X.head()

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
0,Europa,False,B,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0
1,Earth,False,F,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0
2,Europa,False,A,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0
3,Europa,False,A,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0
4,Earth,False,F,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0


In [188]:
# Step 11 - Encode categorical variables

X["HomePlanet"] = X["HomePlanet"].map({
    "Earth": 0,
    "Europa": 1,
    "Mars": 2
})

X["CryoSleep"] = X["CryoSleep"].map({
    False: 0,
    True: 1
})

X["Destination"] = X["Destination"].map({
    "TRAPPIST-1e": 0,
    "55 Cancri e": 1,
    "PSO J318.5-22": 2
})

X["VIP"] = X["VIP"].map({
    False: 0,
    True: 1
})

X["Cabin"] = X["Cabin"].map({
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4,
    "F": 5,
    "G": 6,
    "T": 7
})

X.head()

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
0,1,0,1,0,39.0,0,0.0,0.0,0.0,0.0,0.0
1,0,0,5,0,24.0,0,109.0,9.0,25.0,549.0,44.0
2,1,0,0,0,58.0,1,43.0,3576.0,0.0,6715.0,49.0
3,1,0,0,0,33.0,0,0.0,1283.0,371.0,3329.0,193.0
4,0,0,5,0,16.0,0,303.0,70.0,151.0,565.0,2.0


In [189]:
# Step 12 - Check X after encoding

print(X.dtypes)
print("\nMissing values:")
print(X.isnull().sum())

HomePlanet        int64
CryoSleep         int64
Cabin             int64
Destination       int64
Age             float64
VIP               int64
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
dtype: object

Missing values:
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
dtype: int64


In [190]:
# Step 13 - Train/Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (6954, 11)
X_test shape: (1739, 11)
y_train shape: (6954,)
y_test shape: (1739,)


In [191]:
# Step 14 - Feature Scaling

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

X_train_scaled shape: (6954, 11)
X_test_scaled shape: (1739, 11)


In [192]:
# Step 15 - Feature Selection

from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(
    score_func=f_classif,
    k=10
)

X_train_selected = selector.fit_transform(
    X_train_scaled,
    y_train
)

X_test_selected = selector.transform(
    X_test_scaled
)

print("Original number of features:", X_train.shape[1])
print("Selected number of features:", X_train_selected.shape[1])

Original number of features: 11
Selected number of features: 10


- Bagging and Pasting

In [193]:
# Step 16 - Bagging

bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=SEED),
    n_estimators=100,
    bootstrap=True,
    random_state=SEED
)

bagging_model.fit(
    X_train_selected,
    y_train
)

print("Bagging model trained successfully.")

Bagging model trained successfully.


In [194]:
# Step 18 - Evaluate Bagging

y_pred_bagging = bagging_model.predict(X_test_selected)

bagging_accuracy = accuracy_score(
    y_test,
    y_pred_bagging
)

print(f"Bagging Accuracy: {bagging_accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_bagging))

Bagging Accuracy: 78.09%

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.78      0.78       863
           1       0.78      0.79      0.78       876

    accuracy                           0.78      1739
   macro avg       0.78      0.78      0.78      1739
weighted avg       0.78      0.78      0.78      1739



In [195]:
# Step 19 - Pasting

pasting_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=SEED),
    n_estimators=100,
    bootstrap=False,
    random_state=SEED
)

pasting_model.fit(
    X_train_selected,
    y_train
)

print("Pasting model trained successfully.")

Pasting model trained successfully.


In [196]:
# Step 20 - Evaluate Pasting

y_pred_pasting = pasting_model.predict(X_test_selected)

pasting_accuracy = accuracy_score(
    y_test,
    y_pred_pasting
)

print(f"Pasting Accuracy: {pasting_accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_pasting))

Pasting Accuracy: 73.43%

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.70      0.72       863
           1       0.72      0.76      0.74       876

    accuracy                           0.73      1739
   macro avg       0.74      0.73      0.73      1739
weighted avg       0.73      0.73      0.73      1739



- Random Forests

In [197]:
# Step 21 - Random Forest

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=SEED
)

random_forest_model.fit(
    X_train_selected,
    y_train
)

print("Random Forest model trained successfully.")

Random Forest model trained successfully.


In [198]:
# Step 22 - Evaluate Random Forest

y_pred_rf = random_forest_model.predict(X_test_selected)

rf_accuracy = accuracy_score(
    y_test,
    y_pred_rf
)

print(f"Random Forest Accuracy: {rf_accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

Random Forest Accuracy: 78.32%

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.78      0.78       863
           1       0.79      0.78      0.78       876

    accuracy                           0.78      1739
   macro avg       0.78      0.78      0.78      1739
weighted avg       0.78      0.78      0.78      1739



- Gradient Boosting

In [199]:
# Step 23 - Gradient Boosting

gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=SEED
)

gradient_boosting_model.fit(
    X_train_selected,
    y_train
)

print("Gradient Boosting model trained successfully.")

Gradient Boosting model trained successfully.


In [200]:
# Step 25 - Evaluate Gradient Boosting

y_pred_gb = gradient_boosting_model.predict(
    X_test_selected
)

gb_accuracy = accuracy_score(
    y_test,
    y_pred_gb
)

print(f"Gradient Boosting Accuracy: {gb_accuracy * 100:.2f}%")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_gb
    )
)

Gradient Boosting Accuracy: 78.78%

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.74      0.77       863
           1       0.76      0.84      0.80       876

    accuracy                           0.79      1739
   macro avg       0.79      0.79      0.79      1739
weighted avg       0.79      0.79      0.79      1739



- Adaptive Boosting

In [201]:
# Step 27 - AdaBoost

adaboost_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(
        max_depth=1,
        random_state=SEED
    ),
    n_estimators=100,
    learning_rate=0.1,
    random_state=SEED
)

adaboost_model.fit(
    X_train_selected,
    y_train
)

print("AdaBoost model trained successfully.")

AdaBoost model trained successfully.


In [202]:
# Step 28 - Evaluate AdaBoost

y_pred_ada = adaboost_model.predict(
    X_test_selected
)

ada_accuracy = accuracy_score(
    y_test,
    y_pred_ada
)

print(f"AdaBoost Accuracy: {ada_accuracy * 100:.2f}%")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_ada
    )
)

AdaBoost Accuracy: 75.39%

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.84      0.77       863
           1       0.81      0.67      0.73       876

    accuracy                           0.75      1739
   macro avg       0.76      0.75      0.75      1739
weighted avg       0.76      0.75      0.75      1739



Which model is the best and why?

In [203]:
# Step 29 - Compare all Ensemble Models

results = pd.DataFrame({
    "Model": [
        "Bagging",
        "Pasting",
        "Random Forest",
        "Gradient Boosting",
        "AdaBoost"
    ],
    "Test Accuracy": [
        bagging_accuracy,
        pasting_accuracy,
        rf_accuracy,
        gb_accuracy,
        ada_accuracy
    ]
})

results = results.sort_values(
    by="Test Accuracy",
    ascending=False
).reset_index(drop=True)

results

,Model,Test Accuracy
0,Gradient Boosting,0.787809
1,Random Forest,0.783209
2,Bagging,0.780909
3,AdaBoost,0.753882
4,Pasting,0.734330


In [204]:
# Step 30 - Identify the best model

best_model = results.iloc[0]

print("Best Model:", best_model["Model"])
print(f"Test Accuracy: {best_model['Test Accuracy'] * 100:.2f}%")

Best Model: Gradient Boosting
Test Accuracy: 78.78%


Five ensemble learning methods were evaluated on the Spaceship Titanic dataset: Bagging, Pasting, Random Forest, Gradient Boosting, and AdaBoost.
The results were:
- Bagging: 78.09%
- Pasting: 73.43%
- Random Forest: 78.32%
- Gradient Boosting: 78.78%
- AdaBoost: 75.39%
Gradient Boosting achieved the highest test accuracy of 78.78%, making it the best-performing model among the ensemble methods tested.
Gradient Boosting performed slightly better than Random Forest and Bagging. Pasting had the lowest accuracy, while AdaBoost performed better than Pasting but worse than the other three methods.
Overall, the results show that combining multiple decision trees can provide good predictive performance. For this dataset, Gradient Boosting was the most effective of the ensemble methods tested.